# Data Loading Notebook

This notebook covers loading and displaying basic information about the raw dataset scraped from TW's website.

## Table of Contents

1. [Load data](#Load-data)
2. [Examine each column](#Examine-each-column)
3. [Filter out junior racquets](#filter-out-junior-racquets)
4. [Takeaways and next steps](#takeaways-and-next-steps)

In [1]:
# Import all packages here
import pandas as pd
import numpy as np
from pathlib import Path

## Load data

In [2]:
# Define data directory path and load data- assuming you're in notebooks/
DATA_DIR = Path().cwd().parent / "data"
raw_df = pd.read_csv(DATA_DIR / "raw" / "racquets_raw_2026_03_27.csv")

In [3]:
raw_df.head()

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,head_size,length,strung_weight,...,string_pattern,string_tension,racquet_colors_,racquet_colors_grey/,age,height,weight,other,age_9-10,strung _weight
0,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 2026,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,100 in² / 645.16 cm²,27in / 68.58cm,11.2oz / 318g,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2026,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.tennis-warehouse.com/Babolat_Pure_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2-Pack 2026,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Plus 2026,2.0,2.0,299.0,One of the games most powerful and spin-friend...,100 in² / 645.16 cm²,27.5in / 69.85cm,11.3oz / 320g,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**It might be good to extract the racquet brand name for metadata & filtering in the search workflow.**

Below, I'll look at the shape and columns of the df. It has 398 rows and 37 columns. The additional columns are likely extraneuous "spec" columns that got appended for niche, junior, or special racquets. **The best approach seem to be to just remove these columns as they don't convey useful information for most racquets**.

In [4]:
raw_df.shape

(438, 30)

In [5]:
raw_df.columns

Index(['racquet_url', 'racquet_img', 'racquet_name', 'racquet_rating',
       'racquet_rating_count', 'racquet_price', 'racquet_description',
       'head_size', 'length', 'strung_weight', 'balance', 'swingweight',
       'stiffness', 'beam_width', 'composition', 'power_level', 'stroke_style',
       'swing_speed', 'racquet_colors', 'grip_type', 'string_pattern',
       'string_tension', 'racquet_colors_', 'racquet_colors_grey/', 'age',
       'height', 'weight', 'other', 'age_9-10', 'strung _weight'],
      dtype='object')

Below is a list of each column, it's non-null count, and its data type. The last couple columns are almost entirely NA. There were also clearly some non-standard racquet pages within the brand pages as there are 14 "racquets" that do not have names but that had functioning links. 

**Before doing any further cleaning, I will need to filter for racquets that have a name**.

In [6]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438 entries, 0 to 437
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   racquet_url           438 non-null    object 
 1   racquet_img           418 non-null    object 
 2   racquet_name          418 non-null    object 
 3   racquet_rating        267 non-null    float64
 4   racquet_rating_count  266 non-null    float64
 5   racquet_price         418 non-null    float64
 6   racquet_description   405 non-null    object 
 7   head_size             383 non-null    object 
 8   length                383 non-null    object 
 9   strung_weight         324 non-null    object 
 10  balance               326 non-null    object 
 11  swingweight           325 non-null    float64
 12  stiffness             324 non-null    object 
 13  beam_width            326 non-null    object 
 14  composition           383 non-null    object 
 15  power_level           3

## Examine each column

Below are the descriptive statistics for the numeric and object columns. Notice that some columns that *should* be numeric, like `head_size`, do not appear in the numeric columns' descriptive statistics. This is because all of the spec values were recorded as strings during scraping. 

**I will need to process those columns to convert them into float types**. 

In [7]:
# Numeric summary stats
raw_df.describe()

,racquet_rating,racquet_rating_count,racquet_price,swingweight
count,267.000000,266.000000,418.000000,325.000000
mean,4.717603,9.804511,196.020311,316.378462
std,0.386362,10.836666,92.420871,11.338568
min,2.000000,1.000000,13.950000,270.000000
25%,4.600000,3.000000,127.500000,310.000000
50%,4.800000,6.000000,199.000000,317.000000
75%,5.000000,13.000000,259.000000,325.000000
max,5.000000,59.000000,619.000000,345.000000


In [9]:
# Object summary stats
raw_df.describe(include = ["object"])

,racquet_url,racquet_img,racquet_name,racquet_description,head_size,length,strung_weight,balance,stiffness,beam_width,...,string_pattern,string_tension,racquet_colors_,racquet_colors_grey/,age,height,weight,other,age_9-10,strung _weight
count,438,418,418,405,383,383,324,326,324,326,...,329,323,1,1,56,8,3,5,1,2
unique,438,418,410,398,44,23,32,79,22,93,...,37,23,1,1,11,6,3,3,1,1
top,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,"Dunlop FX Junior 25""",Easy power. Easy Spin. Great Price,100 in² / 645.16 cm²,27in / 68.58cm,11.4oz / 323g,13in / 33.02cm / 4 pts HL,66,23mm / 26mm / 23mm,...,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",50-60 pounds,Gray/Yellow,Grey/ Pink,9-10,50-55 inches / 127cm-140cm,8.1 ounces / 230 grams,String Tension:45-55 poundsSolinco recommends ...,9-10,11.1oz / 315g
freq,1,1,2,2,161,276,56,46,44,35,...,97,77,1,1,20,2,1,3,1,2


Just from the eye-test, there are several columns other than `head_size` that should be numeric. These include `length`, `strung weight`, `balance`, `stiffness`, `beam_width`, `string_pattern`, and `string_tension`. All of these columns (with the exception of `stiffness`)will need to be parsed/preprocessed before being able to be converted to numeric types. Below are the first 5 rows of the data for just those columns. 

In [10]:
raw_df[["head_size", "length", "strung_weight", "balance", "stiffness", "beam_width", "string_pattern", "string_tension"]].head()

,head_size,length,strung_weight,balance,stiffness,beam_width,string_pattern,string_tension
0,100 in² / 645.16 cm²,27in / 68.58cm,11.2oz / 318g,12.99in / 32.99cm / 4 pts HL,66,23mm / 26mm / 23mm,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds
1,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,12.79in / 32.49cm / 6 pts HL,66,21mm / 23mm / 22mm,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,12.79in / 32.49cm / 6 pts HL,66,21mm / 23mm / 22mm,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds
4,100 in² / 645.16 cm²,27.5in / 69.85cm,11.3oz / 320g,12.99in / 32.99cm / 6 pts HL,65,23mm / 26mm / 23mm,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds


## Filter out junior racquets

There are two types of racquets in the data: regular racquets and junior racquets. My initial hunch is that the junior racquets are contributing to the NA values in the data. In the grand context of building a semantic search tool for tennis racquets, I think it is acceptable to exclude junior racquets. Semantic search aims to improve the shopping experience for tennis players who are able to semantically describe what they are looking for in a frame, which would mostly be passionate adult players or higher level juniors who are already using adult racquets.

Given this context, **I will filter out junior racquets from the dataset**.

In [11]:
# Create a new df without jr racquets
explore_df = raw_df.copy()
explore_df["is_junior"] = explore_df["racquet_name"].str.lower().str.contains("junior")

# Compare NA values between full and no jr datasets
ct_na_full = pd.Series(explore_df.isna().sum(), name = "ct_na_full")
pct_na_full = pd.Series(((ct_na_full / len(explore_df))).round(2), name = "pct_na_full")
ct_na_no_jr = pd.Series(explore_df[explore_df["is_junior"] == False].isna().sum(), name = "ct_na_no_jr")
pct_na_no_jr = pd.Series(((ct_na_no_jr / len(explore_df[explore_df["is_junior"] == False]))).round(2), name = "pct_na_no_jr")

na_comparison_df = pd.concat([ct_na_full, pct_na_full, ct_na_no_jr, pct_na_no_jr], axis = 1).reset_index(names = "column")

na_comparison_df["pct_pct_na_reduction"] = np.where(
    na_comparison_df["pct_na_full"] != na_comparison_df["pct_na_no_jr"],
    ((na_comparison_df["pct_na_full"] - na_comparison_df["pct_na_no_jr"]) / na_comparison_df["pct_na_full"]),
    0
)

As expected, removing Jr racquets reduces or completely eliminates NA values for the standard columns and actually boosts the "non-standard" spec columns to 100% NAs (indicating that those columns only exist for Jr racquets).

In [12]:
na_comparison_df.sort_values(by = "pct_pct_na_reduction", ascending = False)

,column,ct_na_full,pct_na_full,ct_na_no_jr,pct_na_no_jr,pct_pct_na_reduction
30,is_junior,20,0.05,0,0.00,1.000000
2,racquet_name,20,0.05,0,0.00,1.000000
5,racquet_price,20,0.05,0,0.00,1.000000
1,racquet_img,20,0.05,0,0.00,1.000000
11,swingweight,113,0.26,10,0.03,0.884615
19,grip_type,112,0.26,9,0.03,0.884615
17,swing_speed,113,0.26,10,0.03,0.884615
16,stroke_style,113,0.26,10,0.03,0.884615
13,beam_width,112,0.26,9,0.03,0.884615
12,stiffness,114,0.26,11,0.03,0.884615


## Takeaways and next steps


1. Get rid of all junior racquets from df

2. Add a racquet_brand column by extracting the brand name from racquet_name

3. Convert the following columns from object to float or int using regex or str logic:
    - Head Size
    - Length
    - Strung Weight
    - Balance
        - Create two columns: racquet_balance_in and racquet_balance_HH_HL
    - Stiffness
    - Beam width
    - String Pattern
        - Create two columns: racquet_mains and racquet_crosses
    - String Tension
        - Create two columns: racquet_tension_lower and racquet_tension_upper

Overall, the scraper did a decent job considering some of the oddities of the TW racquet pages. There are junior racquets present in the data, which aren't relevant and there are several columns that need to be processed into a different data type. 

Since the preprocessing required to fix these errors is relatively minimal, I've decided to use the scraper as-is and just quickly performing the basic preprocessing on the data afterwards.